In [2]:
%reload_ext autoreload
%autoreload 2
import sys
from pathlib import Path
# 查找项目根目录（包含core文件夹的目录）
current_path = Path.cwd()
while current_path.parent != current_path:  # 避免到达根目录
    if (current_path / 'core').exists():
        project_root = current_path
        break
    current_path = current_path.parent
else:
    project_root = Path.cwd()
sys.path.append(str(project_root))
from core.data_loader import DataLoader
from core.synchronizer import DataSynchronizer
from core.spike_sorter import SpikeSorter
from core.quality_controller import QualityController
from core.data_integrator import DataIntegrator
from utils.config_manager import get_config_manager

✅ ipywidgets available - interactive GUI ready


# Main stream

In [2]:
data_directory = fr'F:\ProcessPipeline\testdata\wordfob'
# step 1 读取采集数据
data_loader = DataLoader(data_directory)
## 读取 spikeGLX 
data_loader.load_spikeglx()
neural_data = data_loader.get_spikeglx_data() # imec0.ap
## 读取 MonkeyLogic
data_loader.load_monkeylogic()
monkeylogic_data = data_loader.get_monkeylogic_data()
## 同步数据 与 元数据
sync_data = data_loader.get_sync_data()
metadata = data_loader.get_metadata()
print(sync_data.keys(), metadata.keys())
# step 2 数据同步校准
synchronizer = DataSynchronizer(data_loader)
synchronizer.process_full_synchronization()
export_data = synchronizer.get_export_data()
print(export_data.keys())


2025-09-28 20:35:42 - process_20250928_203534 - INFO - MATLAB引擎初始化成功
2025-09-28 20:35:42 - process_20250928_203534 - INFO - 开始步骤: load_spikeglx - 加载SpikeGLX神经数据和同步信号 (流: imec0.ap)
2025-09-28 20:35:42 - process_20250928_203534 - INFO - 找到SpikeGLX文件夹: F:\ProcessPipeline\testdata\wordfob\NPX_MD241029_exp_g0
2025-09-28 20:35:43 - process_20250928_203534 - INFO - 成功加载nidq同步数据
2025-09-28 20:35:43 - process_20250928_203534 - INFO - 数字通道映射: {'sync': 0, 'trial_start': 3, 'stim_onset': 6}
2025-09-28 20:35:43 - process_20250928_203534 - INFO - 成功加载imec同步数据
2025-09-28 20:35:43 - process_20250928_203534 - INFO - 步骤完成: load_spikeglx (耗时: 0:00:00.411962) - 成功加载SpikeGLX数据和同步信号
2025-09-28 20:35:43 - process_20250928_203534 - INFO - 开始步骤: load_monkeylogic - 加载MonkeyLogic行为数据
2025-09-28 20:35:43 - process_20250928_203534 - INFO - 找到MonkeyLogic文件: F:\ProcessPipeline\testdata\wordfob\241026_MaoDan_YJ_WordLOC.bhv2
2025-09-28 20:35:43 - process_20250928_203534 - INFO - 使用scipy.io成功加载MAT文件
2025-09-28 20:35:43

In [8]:
import h5py 
processed_path = Path(fr'F:\ProcessPipeline\testdata\wordfob\processed')
with h5py.File(processed_path / 'META_wordfob.h5', 'r') as f:
    print(f['sync_info'].keys())
    stim_onset_ms = f['sync_info']['calibrated_onset_ms'][:]
    stim_start_times = f['sync_info']['stim_start_times'][:]
    stim_end_times = f['sync_info']['stim_end_times'][:]
    
    

<KeysViewHDF5 ['calibrated_onset_ms', 'stim_end_times', 'stim_start_times', 'sync_line']>


In [11]:
stim_onset_ms.shape, stim_start_times.shape, stim_end_times.shape

((972,), (972,), (972,))

In [3]:
# step 3 kilosort
neural_data = data_loader.get_spikeglx_data()
sorter = SpikeSorter.from_recording(neural_data)
# 预处理
sorter.preprocess()
# 运行 sorting
sorter.run_kilosort()
# step 5 quality check
processed_path = Path(fr'F:\ProcessPipeline\testdata\wordfob\processed')
ks_output_path = processed_path / "kilosort_output/sorter_output"
spikeglx_path = fr'F:\ProcessPipeline\testdata\wordfob\NPX_MD241029_exp_g0\NPX_MD241029_exp_g0_imec0'
quality_controller = QualityController(kilosort_output_path=ks_output_path, 
    imec_data_path=spikeglx_path)
bc_params = quality_controller.setup_bombcell_params()
bombcell_results = quality_controller.run_quality_control()

2025-09-27 12:49:29 - process_20250927_124929 - INFO - 使用预加载的Recording对象初始化SpikeSorter
2025-09-27 12:49:29 - process_20250927_124929 - INFO - 采样频率: 30000.201673640167 Hz
2025-09-27 12:49:29 - process_20250927_124929 - INFO - 通道数: 384
2025-09-27 12:49:29 - process_20250927_124929 - INFO - 开始步骤: preprocessing - 神经数据预处理
2025-09-27 12:49:29 - process_20250927_124929 - INFO - 使用SpikeInterface Protocol Pipeline进行预处理
2025-09-27 12:49:29 - process_20250927_124929 - INFO - 参数设置 (预处理协议):
  highpass_filter: {'freq_min': 300.0}
  detect_and_remove_bad_channels: {}
  phase_shift: {}
  common_reference: {'operator': 'median', 'reference': 'global'}
2025-09-27 12:49:29 - process_20250927_124929 - INFO - 应用预处理pipeline...
2025-09-27 12:49:43 - process_20250927_124929 - INFO - 移除了11个坏道: ['imec0.ap#AP1' 'imec0.ap#AP191' 'imec0.ap#AP3' 'imec0.ap#AP5'
 'imec0.ap#AP53' 'imec0.ap#AP55' 'imec0.ap#AP57' 'imec0.ap#AP59'
 'imec0.ap#AP61' 'imec0.ap#AP63' 'imec0.ap#AP65']
2025-09-27 12:49:43 - process_20250927_124

write_zarr_recording (workers: 12 processes):   0%|          | 0/94 [00:00<?, ?it/s]

2025-09-27 12:50:44 - process_20250927_124929 - INFO - 步骤完成: preprocessing (耗时: 0:01:14.831012) - 预处理完成
2025-09-27 12:50:44 - process_20250927_124929 - INFO - 开始步骤: kilosort - 运行Kilosort进行尖峰排序
2025-09-27 12:50:44 - process_20250927_124929 - INFO - 输出目录: F:\ProcessPipeline\testdata\wordfob\processed\kilosort_output
2025-09-27 12:50:44 - process_20250927_124929 - INFO - 使用SpikeInterface Protocol Pipeline进行排序
2025-09-27 12:50:44 - process_20250927_124929 - INFO - 参数设置 (排序协议):
  sorter_name: kilosort4
  verbose: True
  progress_bar: True
  remove_existing_folder: True
  nblocks: 15
  Th_learned: 7.0
  n_jobs: 12
  chunk_duration: 4s
2025-09-27 12:50:44 - process_20250927_124929 - INFO - 开始运行排序...


write_binary_recording (workers: 12 processes):   0%|          | 0/94 [00:00<?, ?it/s]

kilosort.run_kilosort:  
kilosort.run_kilosort: Computing preprocessing variables.
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: N samples: 11199687
kilosort.run_kilosort: N seconds: 373.3203903705975
kilosort.run_kilosort: N batches: 187
kilosort.run_kilosort: Preprocessing filters computed in 0.70s; total 0.71s
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage after preprocessing
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:    23.70 %
kilosort.run_kilosort: Mem used:     25.60 %     |      24.44 GB
kilosort.run_kilosort: Mem avail:    71.11 / 95.56 GB
kilosort.run_kilosort: ------------------------------------------------------
kilosort.run_kilosort: GPU usage:    `conda install pynvml` for GPU usage
kilosort.run_kilosort: GPU memory:   24.68 %     |      2.95   /    11.94 GB
kilosort.run_kilosort: Allocated:     0.07 %     |      0.01   /    11.94 GB
kilosort.ru

...

Computing bombcell quality metrics:   0%|          | 0/352 units


Saving GUI visualization data...
GUI visualization data saved to: F:\ProcessPipeline\testdata\wordfob\processed\bombcell\for_GUI\gui_data.pkl
   Generated spatial decay fits: 327/352 units
   Generated amplitude fits: 327/352 units

🏷️ Classifying units (good/MUA/noise/non-soma)...

Generating summary plots...


c:\Users\Gong\miniconda3\envs\dataprocess\Lib\site-packages\bombcell\plot_functions.py:293: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Saving results...
📁 Saving TSV files to Kilosort directory: F:\ProcessPipeline\testdata\wordfob\processed\kilosort_output\sorter_output
All expected metrics were successfully saved.
2025-09-27 12:56:49 - process_20250927_125636 - INFO - Bombcell结果已保存到: F:\ProcessPipeline\testdata\wordfob\processed\bombcell\bombcell_results.json
2025-09-27 12:56:49 - process_20250927_125636 - INFO - 处理结果 (Bombcell质量控制结果):
  total_units: 352
  good_units: 15
  mua_units: 182
  no-somatic_units: 69
  noise_units: 86
  good_unit_ratio: 0.7556818181818181
  qMetric_summary: {'phy_clusterID': {'mean': 175.5, 'std': 101.61323732663968, 'median': 175.5}, 'nSpikes': {'mean': 1503.403409090909, 'std': 1867.1027025940036, 'median': 866.0}, 'nPeaks': {'mean': 1.764525993883792, 'std': 0.5213147972442822, 'median': 2.0}, 'nTroughs': {'mean': 1.1039755351681957, 'std': 0.36036352612480776, 'median': 1.0}, 'waveformDuration_peakTrough': {'mean': 491.23343527013253, 'std': 212.4032210552244, 'median': 566.66666666666

f:\ProcessPipeline\pyneuralpipe\core\quality_controller.py:228: RuntimeWarning: Mean of empty slice
  'mean': float(np.nanmean(values_array)),
c:\Users\Gong\miniconda3\envs\dataprocess\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
f:\ProcessPipeline\pyneuralpipe\core\quality_controller.py:230: RuntimeWarning: All-NaN slice encountered
  'median': float(np.nanmedian(values_array))


In [ ]:
# step 6 写出为 NWB 文件
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from dateutil import tz
from neuroconv.converters import SpikeGLXConverterPipe
from neuroconv.tools.nwb_helpers import get_default_backend_configuration
from neuroconv.utils.dict import load_dict_from_file, dict_deep_update

# Step 1 make raw data converter & convert to NWB on disk
# ================================
folder_path = Path(fr"F:\ProcessPipeline\testdata\wordfob\NPX_MD241029_exp_g0")
converter = SpikeGLXConverterPipe(folder_path=folder_path, verbose=True)

# Extract what metadata we can from the source files
metadata = converter.get_metadata()

# For data provenance we add the time zone information to the conversion
session_start_time = metadata["NWBFile"]["session_start_time"].replace(tzinfo=tz.tzlocal())
metadata["NWBFile"].update(session_start_time=session_start_time)
metadata["Ecephys"]["ElectrodeGroup"][0]["location"] = "MLO"
# Create the in-memory NWBFile object and retrieve a default configuration for the backend
nwbfile = converter.create_nwbfile(metadata=metadata)
backend_configurations = get_default_backend_configuration(
    nwbfile=nwbfile,
    backend="hdf5",
)

# Make any modifications to the configuration in this step, for example...
dataset_configurations = backend_configurations.dataset_configurations
ap_config = dataset_configurations["acquisition/ElectricalSeriesAP/data"]
lf_config = dataset_configurations["acquisition/ElectricalSeriesLF/data"]
lf_config.chunk_shape = (40000, 64)
lf_config.compression_method = "Blosc"
lf_config.compression_options = {"cname": "zstd", "clevel": 6}

ap_config.chunk_shape = (1, 64)
ap_config.buffer_shape = (200000, 384)
ap_config.chunk_shape = (40000, 64)
ap_config.compression_method = "Blosc"
ap_config.compression_options = {"cname": "zstd", "clevel": 6}

output_folder = Path(fr'F:\ProcessPipeline\nwbfiles')
# Choose a path for saving the nwb file and run the conversion
nwbfile_path = output_folder / "test_session250927_Blosc-zstd-6.nwb"
# Correct
conversion_options = {
    "imec0.ap": {"stub_test": False, 
                "iterator_opts":dict(
                display_progress=True  # Show progress bar
                ),
        },
    "imec0.lf": {"stub_test": False,
                "iterator_opts":dict(
                display_progress=True  # Show progress bar
                ),},
}
converter.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata,
                        conversion_options=conversion_options,
                        backend_configuration=backend_configurations)

# Step 2 writing kilosort results into existed nwb_file as processing module
from neuroconv.datainterfaces import KiloSortSortingInterface
from pynwb import NWBHDF5IO

with NWBHDF5IO(nwbfile_path, 'a') as io:
    nwbfile = io.read()

    ks_dir = fr'F:\ProcessPipeline\testdata\wordfob\processed\kilosort_output\sorter_output'
    # # Change the folder_path to the location of the data in your system
    ks_interface = KiloSortSortingInterface(folder_path=ks_dir, verbose=True)

    ks_conversion_options = { 
        'write_as': 'processing',
        'units_name' : 'kilosort4_unit',
    }
    ks_interface.add_to_nwbfile(nwbfile,**ks_conversion_options)
    io.write(nwbfile)

# Step 3 writing trial events into existed nwb_file
import h5py
import numpy as np
import pandas as pd

meta_file_path = fr'F:\ProcessPipeline\testdata\wordfob\processed\META_wordfob.h5'
with h5py.File(meta_file_path, 'r') as f:
    stim_start_times = f['sync_info']['stim_start_times'][:] / 1000 # convert to second
    stim_end_times = f['sync_info']['stim_end_times'][:] / 1000 # convert to second
    trial_valid_idx = f['trial_validation']['trial_valid_idx'][:]
    dataset_valid_idx = f['trial_validation']['dataset_valid_idx'][:]
    eye_track = f['eye_data']['eye_matrix'][:]
    stim_path = f['session_info'].attrs['dataset_path']
image_names = pd.read_csv(stim_path, sep='\t').FileName.values
image_names = np.insert(image_names.values, 0, 'None')
trial_stim_names = image_names[trial_valid_idx]

with NWBHDF5IO(nwbfile_path, "r+") as io:
    nwbfile = io.read()
    # 创建新列
    nwbfile.add_trial_column(
                    name='stim_index',
                    description='stimulus matlab index (1-indexed)'
                )
    nwbfile.add_trial_column(
                    name='stim_name',
                    description='stimulus name'
    )

    # 
    n_trials = len(stim_start_times)
    all_trial_data = [
            {
                'start_time': stim_start_times[i],
                'stop_time': stim_end_times[i],
                'stim_index': int(trial_valid_idx[i]),
                'stim_name': trial_stim_names[i],
                'fix_success': int(dataset_valid_idx[i] != 0),
            }
            for i in range(n_trials)
        ]
    for i, trial_data in enumerate(all_trial_data):
        nwbfile.add_trial(**trial_data)
        
# Step 5 writing eye tracking data into existed nwb_file
n_timetpoint = eye_track.shape[1]
interval = 0.001 # second
eyetrack_start_times = np.asarray(stim_start_times).reshape(-1, 1)
eyetrack_timestamp = eyetrack_start_times + np.arange(n_timetpoint) * interval
eyetrack_timestamp = eyetrack_timestamp.reshape(-1)
eye_track = eye_track.reshape((-1, 2))
from pynwb.behavior import EyeTracking, SpatialSeries
behavior_module = nwbfile.create_processing_module(
    name="behavior", description="Processed behavioral data"
)

right_eye_positions = SpatialSeries(
    name="right_eye_position",
    description="The position of the right eye measured in degrees.",
    data=eye_track,
    timestamps=eyetrack_timestamp,
    reference_frame="center of screen",
    unit="degrees",
)
eye_tracking = EyeTracking(name="EyeTracking", spatial_series=right_eye_positions)
behavior_module.add(eye_tracking)

# Step 6 writing stimulus data into existed nwb_file
from pynwb import NWBHDF5IO
nwb_file_path = fr'F:\ProcessPipeline\nwbfiles\sorting_session250926_Blosc-zstd-6.nwb'

with NWBHDF5IO(nwb_file_path, "r+") as io:
    nwbfile =  io.read()
    from utils.nwb_stim_helper import StimulusImageManager
    stim_manager = StimulusImageManager(stim_path)
    images_series, index_series = stim_manager.add_to_nwb(
        nwbfile, trial_valid_idx, stim_start_times,
        stimulus_name="visual_stimulus_test2"
    )
    io.write(nwbfile)
    
# Step 7 writing custom unit into existed nwb_file

customed_unit_cols = {
    "ks_id" : "kilosort id",
    "unitpos" : "unit position calculated via center of mass",
    "unittpye" : "unit tpye by bombcell, 1=good, 2=mua, 3=non-soma"
}

import pandas as pd
bc_path = Path(fr'F:\ProcessPipeline\testdata\wordfob\processed\bombcell')
bc_qm_df = pd.read_csv(bc_path / 'templates._bc_qMetrics.csv', index_col=0)

from utils.nwb_bombcell_helper import add_bombcell_column_to_units, convert_bombcell_df_to_nwb_format

# 添加列
for metric_name, metric_description in customed_unit_cols.items():
    nwbfile.add_unit_column(
                name=metric_name,
                description=metric_description,
                data=[]  # 将在添加unit时填充
            )
add_bombcell_column_to_units(nwbfile) 
# 转换bombcell
bc_data = convert_bombcell_df_to_nwb_format(bc_qm_df)


# 添加units
for i, spike_times in enumerate(spike_times_list):
    unit_kwargs = {
        'spike_times': spike_times,
        'ks_id' : ...,
        'unitpos': ...,
        'unittpye': ...,
        'waveform': ...,
        }
    unit_kwargs['bc_qm'] = bc_data[i] if i < len(bc_data) else None


# Step 8 update metadata & subject metadata into existed nwb_file
# ================================
# 2.2 session template from yaml & update
metadata_path = fr"F:\ProcessPipeline\pyneuralpipe\config\nwb_template.yaml"
metadata_from_yaml = load_dict_from_file(file_path=metadata_path)
metadata = dict_deep_update(metadata_from_yaml, metadata)

# 2.3 subject template from yaml & update
subject_path = fr"F:\ProcessPipeline\pyneuralpipe\config\FaLaDi.yaml"
subject_from_yaml = load_dict_from_file(file_path=subject_path)
metadata = dict_deep_update(subject_from_yaml, metadata)

# 2.4 update electrode group location
metadata["Ecephys"]["ElectrodeGroup"][0]["location"] = "MLO"


Metadata is valid!
conversion_options is valid!





c:\Users\Gong\miniconda3\envs\dataprocess\Lib\site-packages\hdmf\container.py:889: UserWarning: 32001 compression may not be available on all installations of HDF5. Use of gzip is recommended to ensure portability of the generated HDF5 files.
  self.fields[dataset_name] = data_io_class(data=data, **data_io_kwargs)









                                             

                                       


                            
100%|██████████| 9/9 [12:49<00:00, 10.98s/it] 






                                             

                                       


                                     
100%|██████████| 9/9 [12:56<00:00, 10.98s/it] 




100%|██████████| 1/1 [01:48<00:00, 108.14s/it]


In [15]:
from pynwb import NWBHDF5IO
output_folder = Path(fr'F:\ProcessPipeline\nwbfiles')
# Choose a path for saving the nwb file and run the conversion
nwbfile_path = output_folder / "test_session250927_Blosc-zstd-6.nwb"
with NWBHDF5IO(str(nwbfile_path), 'r') as io:
    nwbfile = io.read()
    ks_unit = nwbfile.processing['ecephys']['kilosort4_unit'].to_dataframe()

### try write trial into nwbfile

In [6]:
import pandas as pd
image_names = pd.read_csv(stim_path, sep='\t').FileName
image_names.values[0]

'body001-Photoroom.jpg'

In [ ]:
import h5py
import numpy as np
import pandas as pd

meta_file_path = fr'F:\ProcessPipeline\testdata\wordfob\processed\META_wordfob.h5'
with h5py.File(meta_file_path, 'r') as f:
    stim_start_times = f['sync_info']['stim_start_times'][:] / 1000 # convert to second
    stim_end_times = f['sync_info']['stim_end_times'][:] / 1000 # convert to second
    trial_valid_idx = f['trial_validation']['trial_valid_idx'][:]
    dataset_valid_idx = f['trial_validation']['dataset_valid_idx'][:]
    eye_track = f['eye_data']['eye_matrix'][:]
    stim_path = f['session_info'].attrs['dataset_path']
image_names = pd.read_csv(stim_path, sep='\t').FileName.values
image_names = np.insert(image_names.values, 0, 'None')
trial_stim_names = image_names[trial_valid_idx]

# # drop out all invalid trials
# valid_stim_start_times = stim_start_times[dataset_valid_idx!=0]
# valid_stim_end_times = stim_end_times[dataset_valid_idx!=0]
# valid_trial_idx = trial_valid_idx[dataset_valid_idx!=0].astype(np.int64)
# valid_eye_track = eye_track[dataset_valid_idx!=0]
# create stim name array
# 
# valid_stim_names = image_names[valid_trial_idx-1]

output_folder = Path(fr'F:\ProcessPipeline\nwbfiles')
# Choose a path for saving the nwb file and run the conversion
nwbfile_path = output_folder / "test_session250927_Blosc-zstd-6.nwb"

with NWBHDF5IO(nwbfile_path, "r+") as io:
    nwbfile = io.read()
    # 创建新列
    nwbfile.add_trial_column(
                    name='stim_index',
                    description='stimulus matlab index (1-indexed)'
                )
    nwbfile.add_trial_column(
                    name='stim_name',
                    description='stimulus name'
    )

    # 
    n_trials = len(stim_start_times)
    all_trial_data = [
            {
                'start_time': stim_start_times[i],
                'stop_time': stim_end_times[i],
                'stim_index': int(trial_valid_idx[i]),
                'stim_name': trial_stim_names[i],
                'fix_success': int(dataset_valid_idx[i] != 0),
            }
            for i in range(n_trials)
        ]
    for i, trial_data in enumerate(all_trial_data):
        nwbfile.add_trial(**trial_data)

SyntaxError: invalid syntax (488680111.py, line 14)

In [ ]:
# behavior eye tracking 
n_timetpoint = eye_track.shape[1]
interval = 0.001 # second
eyetrack_start_times = np.asarray(stim_start_times).reshape(-1, 1)
eyetrack_timestamp = eyetrack_start_times + np.arange(n_timetpoint) * interval
eyetrack_timestamp = eyetrack_timestamp.reshape(-1)
eye_track = eye_track.reshape((-1, 2))
from pynwb.behavior import EyeTracking, SpatialSeries
behavior_module = nwbfile.create_processing_module(
    name="behavior", description="Processed behavioral data"
)

right_eye_positions = SpatialSeries(
    name="right_eye_position",
    description="The position of the right eye measured in degrees.",
    data=eye_track,
    timestamps=eyetrack_timestamp,
    reference_frame="center of screen",
    unit="degrees",
)
eye_tracking = EyeTracking(name="EyeTracking", spatial_series=right_eye_positions)
behavior_module.add(eye_tracking)


(972, 150)

In [ ]:
# write stimulus into nwb file
from pynwb import NWBHDF5IO
nwb_file_path = fr'F:\ProcessPipeline\nwbfiles\sorting_session250926_Blosc-zstd-6.nwb'

with NWBHDF5IO(nwb_file_path, "r+") as io:
    nwbfile =  io.read()
    from utils.nwb_stim_helper import StimulusImageManager
    stim_manager = StimulusImageManager(stim_path)
    images_series, index_series = stim_manager.add_to_nwb(
        nwbfile, valid_trial_idx, valid_stim_start_times,
        stimulus_name="visual_stimulus_test2"
    )
    io.write(nwbfile)

In [ ]:
# import h5py
# nwb_file_path = fr'F:\ProcessPipeline\nwbfiles\sorting_session250926_Blosc-zstd-6.nwb'

# export_filename = fr'F:\ProcessPipeline\nwbfiles\sorting_session250929_Blosc-zstd-6.nwb'

# with NWBHDF5IO(nwb_file_path, mode="r") as read_io:
#     read_nwbfile = read_io.read()
#     # use the pop method to remove the original TimeSeries from the acquisition group
#     read_nwbfile.acquisition.pop("visual_stimulus_images")
#     read_nwbfile.acquisition.pop("visual_stimulus_test2_images")
#     read_nwbfile.acquisition.pop("visual_stimulus_test_images")

#     read_nwbfile.stimulus.pop("visual_stimulus")
#     read_nwbfile.stimulus.pop("visual_stimulus_test2")
#     read_nwbfile.stimulus.pop("visual_stimulus_test")
#     # call the export method to write the modified NWBFile instance to a new file path.
#     # the original file is not modified
#     with NWBHDF5IO(export_filename, mode="w") as export_io:
#         export_io.export(src_io=read_io, nwbfile=read_nwbfile)

In [ ]:
from pynwb import NWBHDF5IO
nwb_file_path = fr'F:\ProcessPipeline\nwbfiles\sorting_session250929_Blosc-zstd-6.nwb'

with NWBHDF5IO(nwb_file_path, "r") as io:
    read_nwbfile = io.read()


dict_keys(['visual_stimulus', 'visual_stimulus_test', 'visual_stimulus_test2'])


### try write customed unit into nwb file 

In [ ]:
customed_unit_cols = {
    "ks_id" : "kilosort id",
    "unitpos" : "unit position calculated via center of mass",
    "unittpye" : "unit tpye by bombcell, 1=good, 2=mua, 3=non-soma"
}

import pandas as pd
bc_path = Path(fr'F:\ProcessPipeline\testdata\wordfob\processed\bombcell')
bc_qm_df = pd.read_csv(bc_path / 'templates._bc_qMetrics.csv', index_col=0)

from utils.nwb_bombcell_helper import add_bombcell_column_to_units, convert_bombcell_df_to_nwb_format

# 添加列
for metric_name, metric_description in customed_unit_cols.items():
    nwbfile.add_unit_column(
                name=metric_name,
                description=metric_description,
                data=[]  # 将在添加unit时填充
            )
add_bombcell_column_to_units(nwbfile) 
# 转换bombcell
bc_data = convert_bombcell_df_to_nwb_format(bc_qm_df)


# 添加units
for i, spike_times in enumerate(spike_times_list):
    unit_kwargs = {'spike_times': spike_times}
    
    unit_kwargs['bc_qm'] = bc_data[i] if i < len(bc_data) else None

    



['phy_clusterID',
 'nSpikes',
 'nPeaks',
 'nTroughs',
 'waveformDuration_peakTrough',
 'spatialDecaySlope',
 'waveformBaselineFlatness',
 'scndPeakToTroughRatio',
 'mainPeakToTroughRatio',
 'peak1ToPeak2Ratio',
 'troughToPeak2Ratio',
 'mainPeak_before_width',
 'mainTrough_width',
 'percentageSpikesMissing_gaussian',
 'percentageSpikesMissing_symmetric',
 'RPV_window_index',
 'fractionRPVs_estimatedTauR',
 'presenceRatio',
 'maxDriftEstimate',
 'cumDriftEstimate',
 'rawAmplitude',
 'signalToNoiseRatio',
 'isolationDistance',
 'Lratio',
 'silhouetteScore',
 'useTheseTimesStart',
 'useTheseTimesStop',
 'maxChannels']

### try export spikesorting processing results in many methods

In [20]:
# try use spikeinterface to write sorting_analyzer to nwb_file
from neuroconv.tools.spikeinterface import write_sorting_analyzer_to_nwbfile
from pathlib import Path
import spikeinterface.full as si

output_folder = Path(fr'F:\ProcessPipeline\nwbfiles')
nwbfile_path = output_folder / "test_session250926_Blosc-zstd-6.nwb"

ks_dir = fr'F:\ProcessPipeline\testdata\wordfob\processed\kilosort_output\sorter_output'
analyzer_dir = fr'F:\ProcessPipeline\testdata\wordfob\processed\sorting_analyzer'
sorting_analyzer = si.load(analyzer_dir)
# write_sorting_analyzer_to_nwbfile(sorting_analyzer=sorting_analyzer, 
#                           nwbfile_path=nwbfile_path, 
#                           write_as='processing',
#                           units_name='ksanalyzer')
sorting_analyzer.sorting.get_property_keys()

['Amplitude', 'ContamPct', 'KSLabel', 'KSLabel_repeat', 'original_cluster_id']

In [ ]:
# try use  KiloSortSortingInterface in neuroconv to conter sorting_dir
from neuroconv.datainterfaces import KiloSortSortingInterface
from pathlib import Path
from pynwb import NWBHDF5IO

output_folder = Path(fr'F:\ProcessPipeline\nwbfiles')
nwbfile_path = output_folder / "test_session250926_Blosc-zstd-6.nwb"

with NWBHDF5IO(nwbfile_path, 'a') as io:
    nwbfile = io.read()

    ks_dir = fr'F:\ProcessPipeline\testdata\wordfob\processed\kilosort_output\sorter_output'
    # # Change the folder_path to the location of the data in your system
    ks_interface = KiloSortSortingInterface(folder_path=ks_dir, verbose=True)

    ks_conversion_options = { 
        'write_as': 'processing',
        'units_name' : 'kilosort4_unit',
    }
    ks_interface.add_to_nwbfile(nwbfile,**ks_conversion_options)
    io.write(nwbfile)



root pynwb.file.NWBFile at 0x3104756328400
Fields:
  acquisition: {
    ElectricalSeriesAP <class 'pynwb.ecephys.ElectricalSeries'>,
    ElectricalSeriesLF <class 'pynwb.ecephys.ElectricalSeries'>,
    EventsNIDQDigitalChannelXD0 <class 'abc.LabeledEvents'>,
    EventsNIDQDigitalChannelXD1 <class 'abc.LabeledEvents'>,
    EventsNIDQDigitalChannelXD4 <class 'abc.LabeledEvents'>,
    EventsNIDQDigitalChannelXD5 <class 'abc.LabeledEvents'>,
    EventsNIDQDigitalChannelXD6 <class 'abc.LabeledEvents'>,
    TimeSeriesNIDQ <class 'pynwb.base.TimeSeries'>
  }
  devices: {
    NIDQBoard <class 'pynwb.device.Device'>,
    NeuropixelsImec0 <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    NeuropixelsImec0 <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'pynwb.ecephys.ElectrodesTable'>
  file_create_date: [datetime.datetime(2025, 9, 26, 17, 14, 4, 31597, tzinfo=tzoffset(None, 28800))]
  identifier: e2e5de0b-da14-49d8-b7ee-0b8d4bf4e7f4
  processing: {
    ecephy

In [18]:
with NWBHDF5IO(nwbfile_path, "r") as io:
    read_nwbfile = io.read()
    # print(read_nwbfile)
    print(read_nwbfile.processing['ecephys']['kilosort4_unit'])

kilosort4_unit pynwb.misc.Units at 0x3104829410768
Fields:
  colnames: ['spike_times' 'unit_name' 'silhouetteScore' 'cumDriftEstimate'
 'waveformDuration_peakTrough' 'troughToPeak2Ratio' 'rawAmplitude'
 'mainTrough_width' 'spatialDecaySlope' 'signalToNoiseRatio'
 'useTheseTimesStart' 'nPeaks' 'mainPeak_before_width' 'peak1ToPeak2Ratio'
 'Lratio' 'fractionRPVs_estimatedTauR' 'maxDriftEstimate'
 'mainPeakToTroughRatio' 'percentageSpikesMissing_gaussian'
 'useTheseTimesStop' 'ContamPct' 'KSLabel_repeat' 'isolationDistance'
 'original_cluster_id' 'RPV_window_index' 'scndPeakToTroughRatio'
 'percentageSpikesMissing_symmetric' 'Amplitude' 'presenceRatio'
 'bc_unitType' 'waveformBaselineFlatness' 'nTroughs' 'KSLabel']
  columns: (
    spike_times_index <class 'hdmf.common.table.VectorIndex'>,
    spike_times <class 'hdmf.common.table.VectorData'>,
    unit_name <class 'hdmf.common.table.VectorData'>,
    silhouetteScore <class 'hdmf.common.table.VectorData'>,
    cumDriftEstimate <class 'hdmf

In [13]:
# - check what happen when wirte as new file
from neuroconv.datainterfaces import KiloSortSortingInterface
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

output_folder = Path(fr'F:\ProcessPipeline\nwbfiles')
nwbfile_path = output_folder / "test_session250926_Blosc-zstd-6.nwb"
ks_dir = fr'F:\ProcessPipeline\testdata\wordfob\processed\kilosort_output\sorter_output'
# Change the folder_path to the location of the data in your system
interface = KiloSortSortingInterface(folder_path=ks_dir, verbose=True)

metadata = interface.get_metadata()
session_start_time = datetime(2020, 1, 1, 12, 30, 0, tzinfo=ZoneInfo("US/Pacific"))
metadata["NWBFile"].update(session_start_time=session_start_time)
output_folder = Path(fr'F:\ProcessPipeline\nwbfiles')
path_to_save_nwbfile = output_folder / "sorting_session250926_Blosc-zstd-6.nwb"
nwbfile_path = f"{path_to_save_nwbfile}"  # This should be something like: "./saved_file.nwb"
ks_conversion_options = { 
        'write_as': 'processing',
        'units_name' : 'kilosort4_unit',
    }
interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata, **ks_conversion_options)

In [17]:
with NWBHDF5IO(path_to_save_nwbfile, "r") as io:
    read_nwbfile = io.read()
    # print(read_nwbfile)
    print(read_nwbfile.processing['ecephys']['kilosort4_unit'])

kilosort4_unit pynwb.misc.Units at 0x3104840010896
Fields:
  colnames: ['spike_times' 'unit_name' 'silhouetteScore' 'cumDriftEstimate'
 'waveformDuration_peakTrough' 'troughToPeak2Ratio' 'rawAmplitude'
 'mainTrough_width' 'spatialDecaySlope' 'signalToNoiseRatio'
 'useTheseTimesStart' 'nPeaks' 'mainPeak_before_width' 'peak1ToPeak2Ratio'
 'Lratio' 'fractionRPVs_estimatedTauR' 'maxDriftEstimate'
 'mainPeakToTroughRatio' 'percentageSpikesMissing_gaussian'
 'useTheseTimesStop' 'ContamPct' 'KSLabel_repeat' 'isolationDistance'
 'original_cluster_id' 'RPV_window_index' 'scndPeakToTroughRatio'
 'percentageSpikesMissing_symmetric' 'Amplitude' 'presenceRatio'
 'bc_unitType' 'waveformBaselineFlatness' 'nTroughs' 'KSLabel']
  columns: (
    spike_times_index <class 'hdmf.common.table.VectorIndex'>,
    spike_times <class 'hdmf.common.table.VectorData'>,
    unit_name <class 'hdmf.common.table.VectorData'>,
    silhouetteScore <class 'hdmf.common.table.VectorData'>,
    cumDriftEstimate <class 'hdmf

In [10]:
ks_interface.get_conversion_options_schema()['properties']['write_ecephys_metadata']

{'default': False,
 'type': 'boolean',
 'description': 'Write electrode information contained in the metadata.'}

In [1]:
from pynwb import read_nwb
filepath = fr'F:\ProcessPipeline\nwbfiles\test_session250926.nwb'
nwbfile = read_nwb(filepath)
nwbfile

ValueError: Unable to read file: 'F:\ProcessPipeline\nwbfiles\test_session250926.nwb'. The file is not recognized as either a valid HDF5 or Zarr NWB file. Please ensure the file exists and contains valid NWB data.

In [16]:
from pathlib import Path
nwb_folder = Path(fr'F:\ProcessPipeline\nwbfiles')
nwb_file = nwb_folder / 'test.nwb'
from neuroconv.tools.spikeinterface import write_sorting_analyzer_to_nwbfile
write_sorting_analyzer_to_nwbfile(sorting_analyzer=sorter.sorting_analyzer, 
                          nwbfile_path=nwb_file)

c:\Users\Gong\miniconda3\envs\spikesort\lib\site-packages\neuroconv\tools\nwb_helpers\_metadata_and_file_helpers.py:318: UserWarning: Unable to remove NWB file located at F:\ProcessPipeline\nwbfiles\test.nwb! Please remove it manually.
  _attempt_cleanup_of_existing_nwbfile(nwbfile_path=nwbfile_path_in)
c:\Users\Gong\miniconda3\envs\spikesort\lib\site-packages\neuroconv\tools\nwb_helpers\_metadata_and_file_helpers.py:330: UserWarning: Unable to remove NWB file located at F:\ProcessPipeline\nwbfiles\test.nwb! Please remove it manually.
  _attempt_cleanup_of_existing_nwbfile(nwbfile_path=nwbfile_path_in)


ValidationError: 'NWBFile' is a required property

Failed validating 'required' in schema:
    {'$schema': 'http://json-schema.org/draft-07/schema#',
     '$id': 'base_metafile.schema.json',
     'title': 'Base schema for the metafile',
     'description': 'Base schema for the metafile',
     'version': '0.1.0',
     'type': 'object',
     'required': ['NWBFile'],
     'properties': {'NWBFile': {'type': 'object',
                                'additionalProperties': False,
                                'required': ['session_start_time'],
                                'properties': {'keywords': {'description': 'Terms '
                                                                           'to '
                                                                           'search '
                                                                           'over',
                                                            'type': 'array',
                                          

In [1]:
# try save into nwb
from neuroconv.datainterfaces import SpikeGLXRecordingInterface
from neuroconv.tools.spikeinterface import write_sorting_analyzer_to_nwbfile
from neuroconv.utils.dict import load_dict_from_file, dict_deep_update
from pathlib import Path
from dateutil import tz
import spikeinterface.exporters as sie 
import spikeinterface.full as si
import spikeinterface.extractors as se
from pynwb import NWBHDF5IO

data_path = Path(fr'F:\ProcessPipeline\testdata\wordfob\NPX_MD241029_exp_g0\NPX_MD241029_exp_g0_imec0')
file_path = 'NPX_MD241029_exp_g0_t0.imec0.ap.bin'
interface = SpikeGLXRecordingInterface(file_path=str(data_path/file_path))

metadata = interface.get_metadata()
# assign local timezone
metadata["NWBFile"]["session_start_time"] = metadata["NWBFile"]["session_start_time"].replace(tzinfo=tz.tzlocal())

# session template from yaml
metadata_path = fr"F:\ProcessPipeline\pyneuralpipe\config\nwb_template.yaml"
metadata_from_yaml = load_dict_from_file(file_path=metadata_path)
# subject template from yaml
subject_path = fr"F:\ProcessPipeline\pyneuralpipe\config\FaLaDi.yaml"
subject_from_yaml = load_dict_from_file(file_path=subject_path)

# update metadata
metadata = dict_deep_update(metadata_from_yaml, metadata)
metadata = dict_deep_update(subject_from_yaml, metadata)

metadata["Ecephys"]["ElectrodeGroup"][0]["location"] = "MLO"
nwbfile = interface.create_nwbfile(metadata=metadata)
import os
ks_dir = fr"F:\ProcessPipeline\testdata\wordfob\kilosort_output\sorter_output"

# 1) 读取 Kilosort4 排序结果
# 大多数版本可用：自动检测KS版本
sorting = se.read_kilosort(ks_dir)
recording = se.read_spikeglx(data_path, stream_name='imec0.ap')
# 2) 可选：先计算波形
# 这样导出时会把均值/标准差波形一并写入 Units
write_sorting_analyzer_to_nwbfile(sorting_analyzer=sorter.sorting_analyzer,
                          recording=recording,
                          nwbfile=nwbfile)

from neuroconv.tools import configure_and_write_nwbfile
# final save
nwb_output_path = fr"F:\ProcessPipeline\nwbfiles\Test_20250925.nwb"
configure_and_write_nwbfile(
    nwbfile, nwbfile_path=nwb_output_path, backend="hdf5"
)


FileNotFoundError: [Errno 2] No such file or directory: 'F:\\ProcessPipeline\\testdata\\wordfob\\kilosort_output\\sorter_output\\spike_times.npy'

In [29]:
from pynwb import NWBHDF5IO

# 文件路径
filepath = fr"F:\ProcessPipeline\nwbfiles\Test_20250925.nwb"

# 以只读模式 ('r') 打开文件
with NWBHDF5IO(filepath, 'r') as io:
    nwbfile = io.read()
    unit_df = nwbfile.units.to_dataframe()
    # waveform = nwbfile.units['waveform_mean'].data[:]

In [87]:
import numpy as np
templates = np.load(fr'F:\ProcessPipeline\testdata\wordfob\sorting_analyzer\extensions\templates\average.npy')
print(templates.shape)
unit_locations = np.load(fr'F:\ProcessPipeline\testdata\wordfob\sorting_analyzer\extensions\unit_locations\unit_locations.npy')
print(unit_locations.shape)
spikes = np.load(fr'F:\ProcessPipeline\testdata\wordfob\sorting_analyzer\sorting\spikes.npy')
print(spikes.shape)
wv = np.load(fr'F:\ProcessPipeline\testdata\wordfob\sorting_analyzer\extensions\waveforms\waveforms.npy')
print(wv.shape)


(370, 90, 375)
(370, 2)
(570690,)
(144545, 90, 11)


In [94]:
unit_locations[0]

array([ 0.        , 32.66473017])

In [98]:
ks_output = Path(fr'F:\ProcessPipeline\testdata\wordfob\kilosort_output\sorter_output')
spike_templates = np.load(ks_output / 'spike_templates.npy')
templates = np.load(ks_output / 'templates.npy')
spike_positions = np.load(ks_output / 'spike_positions.npy')
print(spike_positions.shape)
amplitudes = np.load(ks_output / 'amplitudes.npy')
print(amplitudes[spike_templates==0].mean(axis=0))
spike_positions[spike_templates==0].mean(axis=0)

(570690, 2)
14.117924


array([ 0.    , 32.8137], dtype=float32)

In [ ]:
figures_dict = quality_controller.get_bombcell_figures()
import os
# 创建保存目录
os.makedirs('./output_figures', exist_ok=True)

# 保存所有图形
for key, value in figures_dict.items():
    if isinstance(value, list):
        for i, fig in enumerate(value):
            fig.savefig(f'output_figures/{key}_{i+1}.png', dpi=300, bbox_inches='tight')
    else:
        value.savefig(f'output_figures/{key}.png', dpi=300, bbox_inches='tight')


In [1]:
# try compression
import spikeinterface.extractors as se
job_kwargs = {'n_jobs': 8, 'chunk_duration': '4s', 'progress_bar': True}
spikeglx_path = fr'F:\ProcessPipeline\testdata\wordfob\NPX_MD241029_exp_g0\NPX_MD241029_exp_g0_imec0'
recording = se.read_spikeglx(spikeglx_path, stream_name='imec0.ap',  load_sync_channel=False)
recording.save(folder=fr'F:\ProcessPipeline\testdata\wordfob\NPX_MD241029_exp_g0\NPX_MD241029_exp_g0_imec0.zarr',\
                format='zarr', 
                **job_kwargs)

write_zarr_recording 
engine=process - n_jobs=8 - samples_per_chunk=120,000 - chunk_memory=87.89 MiB - total_memory=703.12 MiB - chunk_duration=4.00s


write_zarr_recording (workers: 8 processes):   0%|          | 0/94 [00:00<?, ?it/s]

ZarrRecordingExtractor: 384 channels - 30000.201674 Hz - 1 segments - 11,199,687 samples 
                        373.32s (6.22 minutes) - int16 dtype - 8.01 GiB

In [4]:
import spikeinterface.full as si
si.load(fr'F:\ProcessPipeline\testdata\wordfob\NPX_MD241029_exp_g0\NPX_MD241029_exp_g0_imec0.zarr', format='zarr')

ZarrRecordingExtractor: 384 channels - 30000.201674 Hz - 1 segments - 11,199,687 samples 
                        373.32s (6.22 minutes) - int16 dtype - 8.01 GiB

In [ ]:
import spikeinterface.preprocessing as spr
zarr_spikeglx_path = fr'F:\ProcessPipeline\testdata\wordfob\NPX_MD241029_exp_g0\NPX_MD241029_exp_g0_imec0.zarr'
recording = se.read_zarr(zarr_spikeglx_path)
freq_min = 300
recording = spr.highpass_filter(recording=recording, freq_min=freq_min)
bad_channel_ids, _ = spr.detect_bad_channels(recording)
if len(bad_channel_ids) > 0:
    recording = recording.remove_channels(bad_channel_ids)
recording = spr.phase_shift(recording)
recording = spr.common_reference(recording, operator="median", reference="global")


In [ ]:
from pathlib import Path
job_kwargs = {'n_jobs': 8, 'chunk_duration': '4s', 'progress_bar': True}
temp_folder = Path(fr'F:\ProcessPipeline\testdata\wordfob\KS_TEMP_zarr')
preprocessed_recording = recording.save(
                folder=str(temp_folder), 
                format='zarr', 
                **job_kwargs
            )

### try external file image writer

In [ ]:
from pynwb import NWBHDF5IO, NWBFile
from pynwb.image import ImageSeries
import numpy as np
from datetime import datetime
import os

# --- 1. 创建基础 NWB 文件 ---
nwbfile = NWBFile(
    session_description='My visual stimulus experiment',
    identifier='subject_01_session_123',
    session_start_time=datetime.now().astimezone()
)

# --- 2. 准备刺激数据 ---
# 假设你的图片放在 'stimuli' 文件夹下
stim_folder = 'stimuli'
image_files = sorted([os.path.join(stim_folder, f) for f in os.listdir(stim_folder) if f.endswith('.png')])

# 关键：为每张图片生成一个时间戳
# 这里的 timestamps 必须是根据你的实验记录生成的真实数据
# 例如，每2秒呈现一张图片，从第5秒开始
timestamps = np.arange(len(image_files)) * 2.0 + 5.0

# --- 3. 创建 ImageSeries 并使用“外部引用”方式 ---
stimulus_images = ImageSeries(
    name='VisualStimuli',
    description='Images presented to the subject during the task.',
    unit='n/a',  # 图像没有单位
    external_file=image_files,  # <-- 关键参数：传入文件路径列表
    timestamps=timestamps,      # <-- 关键参数：传入时间戳
    format='external'           # <-- 明确指定格式为外部引用
)

# --- 4. 将 ImageSeries 添加到 NWB 文件的 stimulus 部分 ---
# 'add_stimulus' 是一个方便的方法，它会将对象放入 nwbfile.stimulus
nwbfile.add_stimulus(stimulus_images)

# --- 5. 写入文件 ---
output_path = "subject_01.nwb"
with NWBHDF5IO(output_path, 'w') as io:
    io.write(nwbfile)

print(f"NWB file saved to: {output_path}")

# --- 6. 如何读取和验证 ---
with NWBHDF5IO(output_path, 'r') as io:
    read_nwbfile = io.read()
    
    # 访问数据
    retrieved_image_series = read_nwbfile.stimulus['VisualStimuli']
    
    # 你可以拿到文件路径和时间戳
    print("Number of presented images:", len(retrieved_image_series.external_file))
    print("Path to the first image:", retrieved_image_series.external_file[0])
    print("Timestamp of the first image:", retrieved_image_series.timestamps[0])
    
    # 如果需要加载并显示第一张图片
    # import matplotlib.pyplot as plt
    # import matplotlib.image as mpimg
    # img = mpimg.imread(retrieved_image_series.external_file[0])
    # plt.imshow(img)
    # plt.show()

# test Config

In [ ]:
config_manager = get_config_manager()
config = config_manager.get_synchronizer_config()
config.get('data_loader')

# test spike sorter

In [ ]:
import spikeinterface.full as si
from utils.config_manager import get_config_manager
# 1. 获取配置管理器并查看当前配置
config_manager = get_config_manager()
config = config_manager.get_config()

print("\n1. 当前 Protocol Pipeline 配置:")
print("预处理协议:", config.spike_sorting_pipeline.preprocessing)
print("排序协议:", config.spike_sorting_pipeline.sorting)
print("后处理协议:", list(config.spike_sorting_pipeline.postprocessing.keys()))

recording = si.read_spikeglx(r'F:\ProcessPipeline\testdata\wordfob\NPX_MD241029_exp_g0\NPX_MD241029_exp_g0_imec0', stream_name='imec0.ap')

# test dataloader

## test load_spikeglx

In [ ]:
# test load_spikeglx
data_loader.load_spikeglx()

In [ ]:
type(data_loader.get_spikeglx_data())

In [ ]:
data_loader.spikeglx_data.get_sampling_frequency()
data_loader.spikeglx_data.get_num_frames()
print("=== 所有属性和方法 ===")
methods = [method for method in dir(data_loader.spikeglx_data) if not method.startswith('_')]
for method in sorted(methods):
    print(f"- {method}")

In [ ]:
data_loader.spikeglx_data.time_to_sample_index(90000)

In [ ]:
data_loader.spikeglx_data.get_duration() 

## test load_monkeylogic

In [ ]:
# test load_monkeylogic
data_loader.load_monkeylogic()

In [ ]:
data_loader.monkeylogic_data['AnalogData']['Eye'][10].shape

In [ ]:
"CodeNumbers" in data_loader.monkeylogic_data.get('BehavioralCodes')

In [ ]:
# try different loading methods
import mat73, h5py
mat_file = fr'F:\ProcessPipeline\testdata\processed\ML_250411_JianJian_PV_OE_MEI.mat'
# mat_dict = mat73.loadmat(mat_file)
h5_file = h5py.File(mat_file, 'r')
# with h5py.File(mat_file, 'r') as f:
#     monkeylogic_data = data_loader._parse_hdf5_mat(f)

In [ ]:
import scipy.io as sio 
mat_file = fr'F:\ProcessPipeline\testdata\processed\ML_250411_JianJian_PV_OE_MEI.mat'
mat = sio.loadmat(mat_file, struct_as_record=True, squeeze_me=True)
mat['trial_ML']['AnalogData'][0]['Eye'].item().shape

In [ ]:
mat['trial_ML']['BehavioralCodes'][0]['CodeTimes'].item().shape

In [ ]:
trials_list = mat['trial_ML']
# trials_list.dtype
trials_list['Trial'].shape
    

In [ ]:
data_loader.monkeylogic_data['BehavioralCodes']['CodeTimes'][-1][-1]

In [ ]:
mat['trial_ML']['AbsoluteTrialStartTime'][-1]

## test validate_data

In [ ]:
data_loader.metadata['spikeglx'].get('duration')

In [ ]:
data_loader.validate_data()
neural_duration = data_loader.metadata['spikeglx'].get('duration', 0)
trial_duration  = data_loader.monkeylogic_data.get('duration', 0)

In [ ]:
neural_duration - trial_duration

In [ ]:
data_loader.validation_results.setdefault('issues', []).extend(['HELLO'])

In [ ]:
data_loader.validation_results

In [ ]:
data_loader.get_sync_data().keys()

# test synchronization

anyway we have many inputs settled down, then we try to do check things

In [ ]:
synchronizer = DataSynchronizer(data_loader)

In [ ]:
synchronizer.process_full_synchronization()

### 1s sync line checking

In [ ]:
# prep IMEC
imSampRate = float(data_loader.sync_data['imec_meta'].get('imSampRate'))
Dcode_imec_all = np.diff(np.squeeze(data_loader.sync_data['imec_sync']))
Dcode_imec = {'CodeLoc':[], 'CodeVal':[],'CodeTime':[]}
Dcode_imec['CodeLoc'] = np.where(Dcode_imec_all > 0)[0]
Dcode_imec['CodeVal'] = Dcode_imec_all[Dcode_imec['CodeLoc']]
Dcode_imec['CodeTime'] = 1000 * Dcode_imec['CodeLoc']/imSampRate
# report
for code in np.unique(Dcode_imec['CodeVal']):
    sum_times = np.sum(Dcode_imec['CodeVal']==code)
    print(f'IMEC Event {code} : {sum_times} Times')
# prep NI
niSampRate = float(data_loader.sync_data['nidq_meta'].get('niSampRate'))
Dcode_ni_all = np.diff(np.squeeze(data_loader.sync_data['nidq_digital']))
np.unique(Dcode_ni_all)
Dcode_ni = {'CodeLoc':[], 'CodeVal':[],'CodeTime':[]}
Dcode_ni['CodeLoc'] = np.where(Dcode_ni_all!=0)[0] + 1
Dcode_ni['CodeVal'] = np.squeeze(data_loader.sync_data['nidq_digital'])[Dcode_ni['CodeLoc']]
Dcode_ni['CodeVal'] = np.insert(Dcode_ni['CodeVal'], 0, 0)
Dcode_ni['CodeTime'] = 1000 * Dcode_ni['CodeLoc']/niSampRate
for code in range(8):
    sum_times = np.sum(np.diff(Dcode_ni['CodeVal'] & (1 << code)) > 0)
    print(f'NI Event {1 << code} : {sum_times} Times')
# sync
ni_sync_loc = np.where(np.diff(Dcode_ni['CodeVal'] & (1 << 0)) > 0)[0]
ni_sync_time = Dcode_ni['CodeTime'][ni_sync_loc]
imec_sync_time = Dcode_imec['CodeTime']
time_err = ni_sync_time - imec_sync_time

In [ ]:
# Visualization
# np.max(np.diff(imec_sync_time)), np.max(np.diff(ni_sync_time))

%matplotlib inline
import matplotlib.pyplot as plt
plt.figure(figsize=(2,2))
plt.plot(time_err, lw=0.5)
plt.show()

### ML - NI alignment

In [ ]:
ML_data = data_loader.monkeylogic_data
ML_codes = ML_data['BehavioralCodes']['CodeNumbers']
onset_times = np.sum([np.sum(ML_code==64) for ML_code in ML_codes])
offset_times = np.sum([np.sum(ML_code==32) for ML_code in ML_codes])

NI_trialloc = np.where(np.diff(Dcode_ni['CodeVal'] & (1 << 3)) > 0)[0] + 1
num_trial = len(NI_trialloc)
NI_trialloc = np.insert(NI_trialloc, num_trial, len(Dcode_ni['CodeVal']))

NI_trial_onsets = np.array([np.sum(np.diff(Dcode_ni['CodeVal'][NI_trialloc[_]:NI_trialloc[_+1]] & (1 << 6)) > 0) for _ in range(num_trial)])
ML_trial_onsets = np.array([np.sum(ML_code==64) for ML_code in ML_codes])

# report
onset_err = NI_trial_onsets - ML_trial_onsets
err_trial = np.where(onset_err!=0)[0]
if err_trial.any():
    print(f'NI & ML not match in Trial {err_trial}, with err {onset_err[err_trial]}!')
else:
    print('NI & ML matches')

In [ ]:
plt.figure(figsize=(2,2))
plt.scatter(np.array(NI_trial_onsets), np.array(ML_trial_onsets))
plt.show()

In [ ]:
# dataset stimulus name
datasets = np.unique(ML_data['UserVars'].get('DatasetName')).tolist()
num_dataset = len(datasets)
if num_dataset==1:
    dataset_name = Path(datasets[0]).name.split('.')[0]
print(dataset_name)

In [ ]:
# Eyetracking for trial validation
valid_stim_idx = np.zeros(onset_times)
valid_dataset_idx = np.zeros(onset_times)
stim_global_idx = 0

ML_stimons = ML_data['VariableChanges']['onset_time']
stim_ondurs = np.unique(ML_stimons)
if len(stim_ondurs)==1: eye_matrix = np.nan * np.zeros((onset_times, stim_ondurs[0], 2))
else: eye_matrix = np.nan * np.zeros((onset_times, np.max(stim_ondurs), 2))
ratio_threshold = 0.999

ML_codenumbers = ML_data['BehavioralCodes']['CodeNumbers']
ML_codetimes = ML_data['BehavioralCodes']['CodeTimes']
ML_imagetrian = ML_data["UserVars"]["Current_Image_Train"]
ML_datasetname = ML_data['UserVars']['DatasetName']
ML_sampleinterv = ML_data['AnalogData']['SampleInterval']
ML_eyedata = ML_data['AnalogData']['Eye']
ML_fixwindow = ML_data['VariableChanges']['fixation_window']
for i_trial in range(num_trial):
    stim_ondur = ML_stimons[i_trial]
    stim_onset_loc = np.where(ML_codenumbers[i_trial]==64)[0]
    cur_imagetrian = ML_imagetrian[i_trial][0:len(stim_onset_loc)]
    dataset_idx = datasets.index(ML_datasetname[i_trial]) + 1
    for stim_idx in range(len(stim_onset_loc)):
        stim_start_time = ML_codetimes[i_trial][stim_onset_loc[stim_idx]]
        stim_end_time = stim_start_time + stim_ondur
        stim_start2end = np.floor(np.arange(stim_start_time, stim_end_time) / ML_sampleinterv[i_trial]).astype(np.int16)
        eye_data = ML_eyedata[i_trial][stim_start2end]
        eye_distance = np.linalg.norm(eye_data**2, axis=1)
        eye_valid_ratio = np.sum(eye_distance<ML_fixwindow[i_trial]) / (stim_ondur)
        if eye_valid_ratio > ratio_threshold:
            valid_stim_idx[stim_global_idx] = cur_imagetrian[stim_idx]
            valid_dataset_idx[stim_global_idx] = dataset_idx
        eye_matrix[stim_global_idx, 0:stim_ondur, :] = eye_data
        stim_global_idx += 1


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
# 创建bin范围
binx = np.arange(-8, 8.5, 0.5)  # -8:0.5:8
biny = np.arange(-8, 8.5, 0.5)  # -8:0.5:8

# 初始化密度图矩阵
density_plot = np.zeros([len(binx)-1, len(biny)-1])

# 
plot_eye = np.squeeze(np.mean(eye_matrix, axis=1))

# 提取x和y坐标
x = plot_eye[:, 0]
y = plot_eye[:, 1]

# 使用histogram2d直接生成二维直方图
density_plot, xedges, yedges = np.histogram2d(x, y, bins=[binx, biny])

# 处理0值避免log10警告
density_plot_log = np.where(density_plot > 0, np.log10(density_plot+1e-10), np.nan)

plt.figure(figsize=(3,3))
plt.imshow(density_plot_log.T, extent=[binx[0]-0.15, binx[-1]-0.15, 
                                       biny[0]-0.15, biny[-1]-0.15], 
           origin='lower', aspect='auto')
plt.xlabel('Eyex')
plt.ylabel('Eyey')
plt.show()

In [ ]:
def shaded_error_bar(data, before_onset_measure=0, color='black', alpha=0.3, 
                     xlabel='Time from event', title='After time calibration'):
    """
    绘制带阴影误差条的图形
    
    Parameters:
    -----------
    data : array-like
        数据矩阵 (n_samples x n_timepoints)
    before_onset_measure : int
        时间偏移量
    color : str
        颜色
    alpha : float
        阴影透明度
    """
    x = np.arange(data.shape[1]) - before_onset_measure
    mean_line = np.mean(data, axis=0)
    std_line = np.std(data, axis=0)
    
    plt.figure(figsize=(3,4))
    plt.plot(x, mean_line, color=color, linewidth=2, label='Mean')
    plt.fill_between(x, 
                     mean_line - std_line, 
                     mean_line + std_line, 
                     alpha=alpha, 
                     color=color,
                     label='±1 STD')
    
    plt.xlabel(xlabel)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
from scipy import signal
from fractions import Fraction
from scipy.stats import zscore

niAiRangeMax = int(data_loader.sync_data['nidq_meta'].get('niAiRangeMax'))
niSampRate = float(data_loader.sync_data['nidq_meta'].get('niSampRate'))
fI2V = niAiRangeMax/32768
AIN = data_loader.sync_data['nidq_analog']*fI2V

ratio = 1000 / niSampRate
frac = Fraction(ratio).limit_denominator()
p = frac.numerator
q = frac.denominator
AIN = np.squeeze(signal.resample_poly(AIN, p, q))

before_onset_measure = 10
after_onset_measure = 50
after_onset_stats = 100
onset_LOC = np.where(np.diff(Dcode_ni['CodeVal'] & (1 << 6)) > 0)[0]
stim_onset_ms = np.floor(Dcode_ni['CodeTime'][onset_LOC]).astype(np.int64)
start_times = stim_onset_ms - before_onset_measure
end_times = stim_onset_ms + after_onset_stats

photodoide = np.array([zscore(AIN[start_time:end_time]) for start_time, end_time in zip(start_times, end_times)])


baseline = photodoide[:, :before_onset_measure].mean()
time_lag = before_onset_measure + after_onset_measure
highline = photodoide[:, np.arange(20)+time_lag].mean()
threshold = 0.1*baseline + 0.9*highline

onset_latency = np.array([np.where(photodoide[_,:] > threshold)[0].min() - before_onset_measure for _ in range(photodoide.shape[0])])
stim_onset_ms = stim_onset_ms + onset_latency

# calibration
start_times = stim_onset_ms - before_onset_measure
end_times = stim_onset_ms + after_onset_stats
calibrated_photodoide = np.array([zscore(AIN[start_time:end_time]) for start_time, end_time in zip(start_times, end_times)])
shaded_error_bar(calibrated_photodoide, before_onset_measure)

# valid calibrated photodoide
valid_calibrated_photodoide = calibrated_photodoide[valid_dataset_idx>0, :]
shaded_error_bar(valid_calibrated_photodoide, before_onset_measure, title='Exclude Non-Look Trial')

In [ ]:
valid_stim = valid_stim_idx[valid_stim_idx>0]
unique_stim, counts = np.unique(valid_stim, return_counts=True)
plt.scatter(unique_stim, counts)
plt.show()


In [ ]:
# photodoide_diff = np.diff(photodoide, axis=1)
# photodoide_diff_abs = np.abs(photodoide_diff)
# diff_maxval = np.diag(photodoide_diff[:, np.argmax(photodoide_diff_abs, axis=1)])
# sign_array = np.sign(diff_maxval)
# for stim_idx in np.where(sign_array==-1)[0]:
#     print('.')
#     photodoide[stim_idx, :] = -photodoide[stim_idx, :]



In [ ]:
plt.figure(figsize=(4,4))
plt.imshow(photodoide, aspect='auto')
plt.show()

In [ ]:
from core.synchronizer import DataSynchronizer
synchronizer = DataSynchronizer(data_loader.get_spikeglx_data(), data_loader.get_monkeylogic_data())
# synchronizer.align_timestamps()
# synchronizer.calibrate_onsets()
# synchronizer.validate_trials()
# synchronizer.get_sync_result()


In [ ]:
import spikeinterface.full as si
spikeglx_folder = fr'F:\#Fewshotlearning\data\raw\250820zhuangzhuang\NPX_ZZ250820_exp2_g6'
stream_names, stream_ids = si.get_neo_streams('spikeglx',spikeglx_folder)
print(stream_names)